# Figshare: globally selected UniversalXAS and Tuned-UniversalXAS

This notebook answers one global question: **which hyperparameter setting works best across all eight elements?** It selects one shared UniversalXAS setting and one shared fine-tuning setting using validation data. It then reports validation and held-out test eta for all eight elements.

The completed 384-run UniversalXAS sweep is reused. Each run saved a `best_macro.pt` checkpoint at the epoch with the best mean validation relative median MSE across all eight elements. This notebook scores those checkpoints consistently, averages each hyperparameter setting over its four seeds, and selects one global setting. It does not pick a different universal head per element.

Fine-tuning starts from that single global UniversalXAS checkpoint. The completed fine-tune sweep is used to select one shared strategy, learning rate, and weight decay from validation results. Then the notebook trains exactly one tuned model per element from the selected shared UniversalXAS checkpoint: **eight final fine-tuning runs total**. The same tuning settings and seed are used for every element.

Targets stay unscaled. Baseline scales are computed from training targets only. Validation selects models and hyperparameters; test metrics are calculated after those choices are fixed. These are exploratory results, not a paper-split reproduction unless the input split files match the published membership.

## Remote notebook setup

Run this notebook on the remote OmniXAS machine that contains the completed sweep folder `figshare_universalxas_refined_grid_v3`. If it is elsewhere, set `OMNIXAS_HEAD_SWEEP_DIR` to that existing folder. The completed fine-tune sweep CSV is read from the existing sweep folder. Final models go to a separate `figshare_universalxas_global_hparam_v1` folder; completed models are reused if the cell is rerun.

In [ ]:
import json
import os
import random
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
repo_env = os.environ.get('OMNIXAS_REPO_ROOT')
if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    candidates = [cwd, *cwd.parents]
    REPO_ROOT = next((p for p in candidates if (p / 'omnixas').is_dir() and (p / 'tutorial_omnixas').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Cannot find OmniXAS. Set OMNIXAS_REPO_ROOT to the repository path.')
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser().resolve()
SPLIT_ROOT = Path(os.environ.get('OMNIXAS_SPLIT_ROOT', DATA_ROOT / 'figshare_141_splits_unscaled')).expanduser().resolve()
FEATURE_RUN_DIR = Path(os.environ.get(
    'OMNIXAS_FEATURE_RUN_DIR',
    REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_pipeline' / 'figshare_141_unscaled_seed42',
)).expanduser().resolve()
FEATURE_ROOT = FEATURE_RUN_DIR / 'features'
SCOUT_DIR = Path(os.environ.get(
    'OMNIXAS_HEAD_SWEEP_DIR', FEATURE_RUN_DIR / 'figshare_universalxas_refined_grid_v3',
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'OMNIXAS_GLOBAL_HPARAM_DIR', FEATURE_RUN_DIR / 'figshare_universalxas_global_hparam_v1',
)).expanduser().resolve()
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Final fit uses one shared configuration and one reproducible seed for all elements.
FINAL_TUNE_SEED = 42
MAX_EPOCHS = 1000
PATIENCE = 60
BATCH_SIZE = 4096
EXPECTED_TUNE_RUNS = len(ELEMENTS)

required = [
    SPLIT_ROOT / 'report.json', SPLIT_ROOT / 'manifest.csv', FEATURE_ROOT,
    FEATURE_RUN_DIR / 'RUN_COMPLETE.json', FEATURE_RUN_DIR / 'best_encoder.ckpt',
    SCOUT_DIR / 'universal_candidate_validation.csv', SCOUT_DIR / 'universal_candidates',
    SCOUT_DIR / 'tuned_candidate_validation.csv',
]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Required input is missing:\n' + '\n'.join(map(str, missing)))
report = json.loads((SPLIT_ROOT / 'report.json').read_text(encoding='utf-8'))
if report.get('intensity_normalization') != 'none (factor 1.0; no xsedge-derived scale)':
    raise ValueError(f'Expected unscaled targets; found {report.get("intensity_normalization")!r}')
manifest = pd.read_csv(SPLIT_ROOT / 'manifest.csv')
if not np.allclose(pd.to_numeric(manifest['target_scale_factor']), 1.0, rtol=0, atol=0):
    raise ValueError('Split manifest contains a target scale other than 1.0')

# Verify ID, feature, and unscaled target row order and dimensions for every split.
data = {}
for element in ELEMENTS:
    for split in SPLITS:
        task = f'{element}_FEFF'
        ids_path = SPLIT_ROOT / 'material_id_and_site' / f'{task}_{split}.txt'
        y_path = SPLIT_ROOT / 'ml_data' / f'{task}_{split}_y.txt'
        x_path = FEATURE_ROOT / f'{task}_{split}_X.txt'
        cached_y_path = FEATURE_ROOT / f'{task}_{split}_y.txt'
        ids = [line.strip() for line in ids_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
        x = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        cached_y = np.atleast_2d(np.loadtxt(cached_y_path, dtype=np.float32))
        rows = manifest[(manifest.element == element) & (manifest.split == split)].sort_values('row_index')
        if rows.material_id_and_site.tolist() != ids:
            raise ValueError(f'Manifest/ID order mismatch for {task} {split}')
        if x.shape != (len(ids), 64) or y.shape != (len(ids), 141) or cached_y.shape != y.shape:
            raise ValueError(f'Unexpected shapes for {task} {split}: X={x.shape}, y={y.shape}, cached-y={cached_y.shape}')
        if not np.isfinite(x).all() or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError(f'Invalid feature/target values for {task} {split}')
        if not np.allclose(y, cached_y, rtol=1e-6, atol=1e-7):
            raise ValueError(f'Feature-cache target mismatch for {task} {split}')
        data[(element, split)] = {'ids': ids, 'X': x, 'y': y}

# Train-only baseline mean and median MSE define the loss scale and eta denominator.
TRAIN_MEANS, LOSS_SCALES = {}, {}
for element in ELEMENTS:
    y_train = data[(element, 'train')]['y']
    train_mean = y_train.mean(axis=0, keepdims=True)
    baseline_errors = np.mean((y_train - train_mean) ** 2, axis=1)
    TRAIN_MEANS[element] = train_mean.astype(np.float32)
    LOSS_SCALES[element] = float(np.median(baseline_errors))
    if not np.isfinite(LOSS_SCALES[element]) or LOSS_SCALES[element] <= 0:
        raise ValueError(f'Invalid training-only baseline scale for {element}')

OLD_UNIVERSAL = FEATURE_RUN_DIR / 'heads' / 'universalXAS' / 'best.pt'
OLD_TUNED = {e: FEATURE_RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{e}_FEFF' / 'best.pt' for e in ELEMENTS}
if not OLD_UNIVERSAL.is_file() or not all(p.is_file() for p in OLD_TUNED.values()):
    raise FileNotFoundError('Existing UniversalXAS/Tuned-UniversalXAS comparison checkpoints are missing.')
scout_candidates = pd.read_csv(SCOUT_DIR / 'universal_candidate_validation.csv')
if not {'experiment_id', 'seed', 'loss_balance_power', 'dropout', 'learning_rate'}.issubset(scout_candidates.columns):
    raise ValueError('Universal candidate CSV is missing required hyperparameter columns.')
experiment_ids = sorted(scout_candidates.experiment_id.dropna().unique().tolist())
if len(experiment_ids) != 384:
    raise ValueError(f'Expected all 384 completed universal experiments; found {len(experiment_ids)}.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
provenance = {
    'dataset': str(SPLIT_ROOT), 'feature_run': str(FEATURE_RUN_DIR), 'reused_universal_sweep': str(SCOUT_DIR),
    'new_output': str(OUTPUT_DIR), 'elements': ELEMENTS,
    'universal_selection': 'minimum mean validation relative median spectrum MSE across eight elements, averaged over seeds; use that setting\'s best macro checkpoint',
    'final_fine_tune_seed': FINAL_TUNE_SEED, 'final_fine_tune_count': EXPECTED_TUNE_RUNS, 'target_policy': 'unscaled; no xsedge-derived normalization',
    'fine_tune_setting_selection': 'one shared strategy, learning rate, and weight decay minimizing mean validation relative median MSE over elements and prior source candidates; final refit seed is fixed to 42',
    'device': str(DEVICE),
}
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print(f'Repository: {REPO_ROOT}')
print(f'Unscaled splits: {SPLIT_ROOT}')
print(f'Reused completed universal sweep: {SCOUT_DIR}')
print(f'New results: {OUTPUT_DIR}')
print(f'Device: {DEVICE}; universal candidates to score: {len(experiment_ids)}; fine-tune runs: {EXPECTED_TUNE_RUNS}')

## Select one UniversalXAS setting for all elements

The previous sweep saved a `best_macro.pt` for each configuration, so every candidate here is one shared head at one common epoch. We score that head on every element’s validation set. First, average its relative median MSE across the eight elements; then average that score across the four seeds for each hyperparameter setting. This avoids combining eight different element-specific epochs into a model that never existed.

In [ ]:
from omnixas.model.m3gnet_xas import XASSpectralHead


def load_head(checkpoint):
    saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
    model = XASSpectralHead(dropout=saved.get('dropout', 0.10)).to(DEVICE)
    model.load_state_dict(saved['state_dict'], strict=True)
    return model.eval(), saved


def spectrum_errors(model, element, split):
    item = data[(element, split)]
    x = torch.as_tensor(item['X'], dtype=torch.float32, device=DEVICE)
    y = torch.as_tensor(item['y'], dtype=torch.float32, device=DEVICE)
    model.eval()
    with torch.inference_mode():
        return (model(x) - y).square().mean(dim=1).cpu().numpy()


def metrics_for(element, split, errors):
    y = data[(element, split)]['y']
    baseline = float(np.median(np.mean((y - TRAIN_MEANS[element]) ** 2, axis=1)))
    median_mse = float(np.median(errors))
    return {
        'n_spectra': len(errors), 'aggregate_mse': float(np.mean(errors)),
        'median_spectrum_mse': median_mse, 'baseline_median_mse': baseline,
        'eta': baseline / max(median_mse, 1e-12),
        'relative_median_mse': median_mse / LOSS_SCALES[element],
    }


universal_candidate_rows = []
for experiment_id in tqdm(experiment_ids, desc='Score shared UniversalXAS checkpoints'):
    checkpoint = SCOUT_DIR / 'universal_candidates' / experiment_id / 'best_macro.pt'
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Expected completed macro checkpoint is missing: {checkpoint}')
    model, saved = load_head(checkpoint)
    per_element = {}
    for element in ELEMENTS:
        per_element[element] = metrics_for(element, 'val', spectrum_errors(model, element, 'val'))
    universal_candidate_rows.append({
        'experiment_id': experiment_id, 'checkpoint': str(checkpoint),
        'seed': int(saved['seed']), 'epoch': int(saved['epoch']),
        'loss_balance_power': float(saved['loss_balance_power']), 'dropout': float(saved['dropout']),
        'learning_rate': float(saved['learning_rate']),
        'mean_val_relative_median_mse': float(np.mean([per_element[e]['relative_median_mse'] for e in ELEMENTS])),
        'mean_val_eta': float(np.mean([per_element[e]['eta'] for e in ELEMENTS])),
        **{f'{e}_val_eta': per_element[e]['eta'] for e in ELEMENTS},
    })
universal_scored = pd.DataFrame(universal_candidate_rows)
universal_scored.to_csv(OUTPUT_DIR / 'universal_macro_validation_candidates.csv', index=False)

# Choose hyperparameters by their mean across seeds, then use the best-scoring seed's shared checkpoint.
UNIVERSAL_HPARAMS = ['loss_balance_power', 'dropout', 'learning_rate']
universal_setting_summary = (
    universal_scored.groupby(UNIVERSAL_HPARAMS, as_index=False)
    .agg(mean_val_relative_median_mse=('mean_val_relative_median_mse', 'mean'),
         std_across_seeds=('mean_val_relative_median_mse', 'std'),
         mean_eta=('mean_val_eta', 'mean'), n_seeds=('seed', 'nunique'))
    .sort_values('mean_val_relative_median_mse', ascending=True).reset_index(drop=True)
)
if not (universal_setting_summary.n_seeds == 4).all():
    raise ValueError('Each universal hyperparameter setting must have all four completed seeds.')
universal_setting_summary.to_csv(OUTPUT_DIR / 'universal_global_hyperparameter_summary.csv', index=False)
best_universal_setting = universal_setting_summary.iloc[0]
mask = np.ones(len(universal_scored), dtype=bool)
for column in UNIVERSAL_HPARAMS:
    mask &= np.isclose(universal_scored[column].to_numpy(float), float(best_universal_setting[column]))
selected_universal_run = universal_scored.loc[mask].sort_values('mean_val_relative_median_mse').iloc[0]
UNIVERSAL_CHECKPOINT = Path(selected_universal_run.checkpoint)
pd.DataFrame([{
    **{key: float(best_universal_setting[key]) for key in UNIVERSAL_HPARAMS},
    'mean_val_relative_median_mse_across_seeds': float(best_universal_setting.mean_val_relative_median_mse),
    'selected_seed': int(selected_universal_run.seed), 'selected_epoch': int(selected_universal_run.epoch),
    'selected_checkpoint': str(UNIVERSAL_CHECKPOINT),
}]).to_csv(OUTPUT_DIR / 'selected_global_universal.json.csv', index=False)
print('One global UniversalXAS configuration selected on validation:')
display(universal_setting_summary.head(10).round(5))
print('Selected shared checkpoint:', UNIVERSAL_CHECKPOINT)
print('Selected run seed/epoch:', int(selected_universal_run.seed), int(selected_universal_run.epoch))

## Choose one shared fine-tuning configuration, then fit eight final models

The prior fine-tuning sweep tested many settings on validation. This cell averages each setting’s relative median MSE across the eight elements and the source checkpoints tested for each element, then chooses one shared strategy, learning rate, and weight decay. It uses that setting to train exactly one tuned head per element, all initialized from the same globally selected UniversalXAS checkpoint. No element gets a separately selected configuration.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cpu_state_dict(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}


# Select one global tuning recipe from the already completed validation sweep.
# The old sweep used several candidate source checkpoints per element; average those
# within each element first so each element contributes equal weight to the recipe.
prior_tuned = pd.read_csv(SCOUT_DIR / 'tuned_candidate_validation.csv')
required_cols = {'element', 'strategy', 'learning_rate', 'weight_decay', 'val_median_mse'}
if not required_cols.issubset(prior_tuned.columns):
    raise ValueError(f'Prior fine-tune CSV is missing columns: {sorted(required_cols - set(prior_tuned.columns))}')
prior_tuned = prior_tuned[prior_tuned.element.isin(ELEMENTS)].copy()
prior_tuned['relative_median_mse'] = prior_tuned.apply(
    lambda row: float(row.val_median_mse) / LOSS_SCALES[row.element], axis=1
)
TUNED_HPARAMS = ['strategy', 'learning_rate', 'weight_decay']
per_element_recipe_scores = (
    prior_tuned.groupby(TUNED_HPARAMS + ['element'], as_index=False)
    .agg(element_mean_relative_median_mse=('relative_median_mse', 'mean'),
         source_candidates=('relative_median_mse', 'size'))
)
tuned_setting_summary = (
    per_element_recipe_scores.groupby(TUNED_HPARAMS, as_index=False)
    .agg(mean_val_relative_median_mse=('element_mean_relative_median_mse', 'mean'),
         std_across_elements=('element_mean_relative_median_mse', 'std'),
         n_elements=('element', 'nunique'))
    .sort_values('mean_val_relative_median_mse', ascending=True).reset_index(drop=True)
)
if not (tuned_setting_summary.n_elements == len(ELEMENTS)).all():
    raise ValueError('Prior sweep does not contain every fine-tune configuration for all eight elements.')
tuned_setting_summary.to_csv(OUTPUT_DIR / 'tuned_global_hyperparameter_summary.csv', index=False)
best_tuned_setting = tuned_setting_summary.iloc[0]
selected_recipe = {key: best_tuned_setting[key] for key in TUNED_HPARAMS}
selected_recipe['seed_for_final_refit'] = FINAL_TUNE_SEED
pd.DataFrame([selected_recipe | {
    'mean_val_relative_median_mse_in_prior_sweep': float(best_tuned_setting.mean_val_relative_median_mse),
    'selection_source': str(SCOUT_DIR / 'tuned_candidate_validation.csv'),
}]).to_csv(OUTPUT_DIR / 'selected_global_tuned_configuration.csv', index=False)
print('One shared fine-tuning configuration selected from prior validation results:')
print(selected_recipe)
print(f"Mean relative median MSE across elements: {best_tuned_setting.mean_val_relative_median_mse:.6f}")


def train_tuned(element):
    strategy = str(best_tuned_setting.strategy)
    learning_rate = float(best_tuned_setting.learning_rate)
    weight_decay = float(best_tuned_setting.weight_decay)
    seed = FINAL_TUNE_SEED
    source, source_saved = load_head(UNIVERSAL_CHECKPOINT)
    run_id = (f'final_{strategy}_lr{learning_rate:g}_wd{weight_decay:g}_seed{seed}'
              .replace('.', 'p').replace('-', 'm'))
    out_dir = OUTPUT_DIR / 'selected_tuned_models' / run_id / element
    checkpoint = out_dir / 'best.pt'
    history_path = out_dir / 'validation_history.csv'
    if checkpoint.is_file() and history_path.is_file():
        return checkpoint
    out_dir.mkdir(parents=True, exist_ok=True)
    head = source
    output_layer_index = len(head) - 2
    last_block_start = output_layer_index - 4
    if strategy == 'full_head':
        trainable = list(head.parameters())
        for parameter in trainable:
            parameter.requires_grad_(True)
    elif strategy == 'output_layer_only':
        for parameter in head.parameters():
            parameter.requires_grad_(False)
        output_layer = head[output_layer_index]
        for parameter in output_layer.parameters():
            parameter.requires_grad_(True)
        trainable = list(output_layer.parameters())
    elif strategy == 'last_block':
        for parameter in head.parameters():
            parameter.requires_grad_(False)
        for module_index in range(last_block_start, output_layer_index + 1):
            for parameter in head[module_index].parameters():
                parameter.requires_grad_(True)
        trainable = [parameter for parameter in head.parameters() if parameter.requires_grad]
    else:
        raise ValueError(f'Unknown fine-tuning strategy: {strategy}')

    set_seed(seed)
    optimizer = torch.optim.Adam(trainable, lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=16, min_lr=1e-7
    )
    train = data[(element, 'train')]
    val = data[(element, 'val')]
    train_x = torch.as_tensor(train['X'], dtype=torch.float32, device=DEVICE)
    train_y = torch.as_tensor(train['y'], dtype=torch.float32, device=DEVICE)
    val_x = torch.as_tensor(val['X'], dtype=torch.float32, device=DEVICE)
    val_y = torch.as_tensor(val['y'], dtype=torch.float32, device=DEVICE)
    scale = LOSS_SCALES[element]
    best_relative, stale, history = float('inf'), 0, []
    progress = tqdm(range(MAX_EPOCHS), desc=f'{element} {run_id}', unit='epoch')
    for epoch in progress:
        if strategy == 'full_head':
            head.train()
        elif strategy == 'last_block':
            head.eval()
            head[last_block_start + 1].train()
            head[last_block_start + 3].train()
        else:
            head.eval()
        order = torch.randperm(len(train_x), device=DEVICE)
        for indices in order.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            loss = (head(train_x[indices]) - train_y[indices]).square().mean() / scale
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            val_errors = (head(val_x) - val_y).square().mean(dim=1).cpu().numpy()
        val_median = float(np.median(val_errors))
        relative = val_median / scale
        history.append({'epoch': epoch + 1, 'val_median_mse': val_median, 'val_relative_median_mse': relative})
        if relative < best_relative:
            best_relative, stale = relative, 0
            baseline = float(np.median(np.mean((val['y'] - TRAIN_MEANS[element]) ** 2, axis=1)))
            torch.save({
                'state_dict': cpu_state_dict(head), 'dropout': source_saved.get('dropout', 0.10),
                'element': element, 'source_checkpoint': str(UNIVERSAL_CHECKPOINT),
                'source_experiment_id': source_saved['experiment_id'],
                'source_seed': int(source_saved['seed']), 'strategy': strategy,
                'learning_rate': learning_rate, 'weight_decay': weight_decay, 'seed': seed,
                'epoch': epoch + 1, 'val_median_mse': val_median,
                'val_relative_median_mse': relative, 'val_eta': baseline / max(val_median, 1e-12),
            }, checkpoint)
        else:
            stale += 1
        scheduler.step(relative)
        progress.set_postfix(val_rel=f'{relative:.4f}', best=f'{best_relative:.4f}')
        if stale >= PATIENCE:
            break
    progress.close()
    pd.DataFrame(history).to_csv(history_path, index=False)
    return checkpoint


selected_tuned_rows = []
for element in ELEMENTS:
    checkpoint = train_tuned(element)
    saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
    selected_tuned_rows.append({
        'element': element, 'strategy': saved['strategy'], 'learning_rate': saved['learning_rate'],
        'weight_decay': saved['weight_decay'], 'seed': saved['seed'], 'epoch': saved['epoch'],
        'val_median_mse': saved['val_median_mse'],
        'val_relative_median_mse': saved['val_relative_median_mse'],
        'val_eta': saved['val_eta'], 'checkpoint': str(checkpoint),
    })
selected_tuned_rows = pd.DataFrame(selected_tuned_rows)
if len(selected_tuned_rows) != EXPECTED_TUNE_RUNS:
    raise RuntimeError(f'Expected {EXPECTED_TUNE_RUNS} final tuned heads; found {len(selected_tuned_rows)}')
selected_tuned_rows.to_csv(OUTPUT_DIR / 'selected_global_tuned_models.csv', index=False)
print(f'Trained/reused exactly {len(selected_tuned_rows)} tuned heads with the same configuration.')
display(selected_tuned_rows[['element', 'strategy', 'learning_rate', 'weight_decay', 'seed', 'epoch', 'val_eta']].round(4))

## Final validation and test comparison

The selected UniversalXAS checkpoint is one shared head for all elements. The selected fine-tuning strategy, learning rate, and weight decay are shared too; the eight adapted heads differ only because each is trained on its element’s spectra. The test set is used below for final reporting and does not affect either selection.

In [ ]:
def score_checkpoint(element, variant, checkpoint, split):
    model, _ = load_head(checkpoint)
    return {'element': element, 'variant': variant, 'evaluation_split': split,
            **metrics_for(element, split, spectrum_errors(model, element, split))}


final_rows = []
for element in ELEMENTS:
    tuned_checkpoint = Path(selected_tuned_rows.loc[selected_tuned_rows.element == element, 'checkpoint'].iloc[0])
    candidates = [
        ('Existing UniversalXAS', OLD_UNIVERSAL),
        ('Existing Tuned-UniversalXAS', OLD_TUNED[element]),
        ('Global-sweep UniversalXAS', UNIVERSAL_CHECKPOINT),
        ('Global-sweep Tuned-UniversalXAS', tuned_checkpoint),
    ]
    for variant, checkpoint in candidates:
        for split in ['val', 'test']:
            final_rows.append(score_checkpoint(element, variant, checkpoint, split))
final_metrics = pd.DataFrame(final_rows)
final_metrics.to_csv(OUTPUT_DIR / 'global_universal_tuned_validation_test_metrics.csv', index=False)

VARIANTS = [
    'Existing UniversalXAS', 'Existing Tuned-UniversalXAS',
    'Global-sweep UniversalXAS', 'Global-sweep Tuned-UniversalXAS',
]
for split in ['val', 'test']:
    subset = final_metrics[final_metrics.evaluation_split == split]
    table = subset.pivot(index='element', columns='variant', values='eta').reindex(index=ELEMENTS, columns=VARIANTS)
    table.loc['Mean across elements'] = table.mean(axis=0)
    table['Mean across elements'] = table[VARIANTS].mean(axis=1)
    table.index.name = 'element'
    table.to_csv(OUTPUT_DIR / f'eta_{split}_global_summary.csv')
    print(f'{split.upper()} eta (higher is better; equal element weighting):')
    display(table.round(3))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, split, title in zip(axes, ['val', 'test'], ['Validation eta', 'Test eta']):
    subset = final_metrics[final_metrics.evaluation_split == split]
    plot_data = subset.pivot(index='element', columns='variant', values='eta').reindex(index=ELEMENTS, columns=VARIANTS)
    plot_data.plot(kind='bar', ax=ax, rot=0, width=0.82)
    ax.set_title(title)
    ax.set_xlabel('Element')
    ax.set_ylabel('Eta (higher is better)')
    ax.grid(axis='y', alpha=0.25)
    if ax.get_legend() is not None:
        ax.get_legend().remove()
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08))
fig.suptitle('One globally selected UniversalXAS and fine-tuning configuration', y=1.16)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'global_universal_tuned_eta_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
print(f'Global UniversalXAS setting: alpha={best_universal_setting.loss_balance_power:g}, '
      f'dropout={best_universal_setting.dropout:g}, lr={best_universal_setting.learning_rate:g}; '
      f'seed={int(selected_universal_run.seed)}')
print(f'Global tuned setting: {best_tuned_setting.strategy}, lr={best_tuned_setting.learning_rate:g}, '
      f'weight_decay={best_tuned_setting.weight_decay:g}, seed={int(best_tuned_setting.seed)}')
print(f'All outputs saved to {OUTPUT_DIR}')

## Interpretation

The headline comparison is the **mean across elements** row, with each element weighted equally. The UniversalXAS setting is chosen by mean validation relative median MSE across elements and seeds. The tuned setting uses one shared strategy, learning rate, weight decay, and seed chosen by the same validation criterion; the adapted checkpoint is still trained separately on each element’s training spectra.

Use test eta once as the final comparison. If a tuned model loses to UniversalXAS for an element, that is a real result: a shared fine-tuning recipe does not guarantee every element benefits. Keep the saved candidate tables and `provenance.json` with any result you report.